# RK1828 FastSAM-s Instance Segmentation Demo

[FastSAM](https://github.com/CASIA-IVA-Lab/FastSAM) (CASIA) is a YOLOv8-seg-based
**promptable instance segmentation** model. This demo uses the "segment
everything" mode: one forward pass finds every object and its mask.

The model has two outputs:

| Output | Shape | Meaning |
|---|---|---|
| `output0` | `[1,37,8400]` | cx, cy, w, h, confidence + 32 mask coefficients (decoded) |
| `output1` | `[1,32,160,160]` | mask prototypes |

Post-processing: confidence filter -> NMS -> `sigmoid(coeffs @ prototypes)`.

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Pick an image |
| 3 | Pre-processing (letterbox) |
| 4 | RKNN inference |
| 5 | Post-processing (NMS + mask assembly) and visualization |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


This demo downloads the FastSAM-s model and sample images.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("fastsam-s-rknn3-rk1828", "fastsam_s.rknn", "models/fastsam/fastsam_s.rknn"),
    ("fastsam-s-rknn3-rk1828", "fastsam_s.weight", "models/fastsam/fastsam_s.weight"),
    ("yolo26-segment-rknn3-rk1828", "model/bus.jpg", "datasets/bus.jpg"),
    ("mobilenet_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Pick an image

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="Image:", layout=widgets.Layout(width="60%"))
w_conf = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05,
                             description="Conf:", style={"description_width": "80px"})
display(w_image, w_conf)
print("Then keep running downwards ▼")

## 3. Pre-processing: letterbox

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import letterbox, to_input_array, setup_cjk_font

setup_cjk_font()
img = Image.open(w_image.value).convert("RGB")
img_lb, SCALE, PAD_X, PAD_Y = letterbox(img, 640)
x = to_input_array(img_lb)
print(f"original {img.size} -> input {x.shape}, scale={SCALE:.3f}, pad=({PAD_X},{PAD_Y})")

# show every pre-processing stage: original -> letterbox -> input tensor
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, title in [
    (axes[0], img, f"1. original {img.size[0]}x{img.size[1]}"),
    (axes[1], img_lb, "2. letterbox 640x640\n(aspect resize + gray pad)"),
    (axes[2], x[0], f"3. model input {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN inference

In [ ]:
import time
from rknn3 import Rknn3Model

model = Rknn3Model("models/fastsam/fastsam_s.rknn")
print("outputs:", model.output_attrs)

_ = model.infer([x])
t0 = time.time()
outputs = model.infer([x])
print(f"inference time: {(time.time()-t0)*1000:.1f} ms")

## 5. Post-processing: NMS + mask assembly

`sigmoid(coeffs @ prototypes)` yields 160x160 masks; upscale to input size,
crop by box, then map back to the original image. Fragment masks smaller
than 0.2% of the image are dropped.

In [ ]:
import numpy as np
from vision_post import decode_fastsam, scale_masks, unletterbox_boxes

boxes, scores, masks = decode_fastsam(outputs, conf_thresh=w_conf.value)
masks_full = scale_masks(masks, boxes, (img.height, img.width), SCALE, PAD_X, PAD_Y)
boxes = unletterbox_boxes(boxes, SCALE, PAD_X, PAD_Y, *img.size)

ratio = (masks_full > 0.5).mean(axis=(1, 2))
keep = ratio > 0.002
boxes, scores, masks_full = boxes[keep], scores[keep], masks_full[keep]
print(f"{len(boxes)} objects (after fragment filtering)")

## 6. Visualization

In [ ]:
from vision_post import draw_boxes, draw_masks

overlay = draw_masks(img, masks_full)
result = draw_boxes(overlay, boxes, np.zeros(len(boxes)), scores)
model.close()
result

## Summary

- FastSAM is **class-agnostic**: it only tells "is this an object" plus the
  mask, with no class label.
- Mask = `sigmoid(32 coefficients x 32-channel prototypes)` — the standard
  approach of the YOLOv8-seg family.